# Reproduction fidèle de l'expérience NASA B0018 (fichiers originaux MATLAB)

Ce notebook conserve **l'intégralité du protocole de l'original fourni** : quatre batteries NASA `.mat`, chargement de tous les échantillons de décharge, les sept caractéristiques comprenant `capacity`, normalisation MinMax sur les échantillons de développement, `train_test_split(...test_size=0.2,random_state=0)`, Conv1D(64)–BiLSTM(100)–attention, Adam, MSE, 30 epochs et calcul des métriques **après moyenne des prédictions par cycle**.

Ajouts techniques seulement : clone GitHub, récupération contrôlée des `.mat`, Google Drive, un modèle Keras complet (optimiseur compris) sauvegardé après chaque epoch et une correction de forme `(N,7,1)` nécessaire pour `predict` dans Keras moderne. Les sorties enregistrées des anciens notebooks sont effacées pour ne pas les confondre avec une nouvelle exécution.

**Attention scientifique :** le protocole historique réutilise `capacity` dans les entrées alors qu'elle définit le SOH et réajuste le scaler sur la batterie testée. Il présente donc une fuite de cible. Les scores obtenus ici servent à **reproduire et auditer** les chiffres historiques, pas à établir des performances de généralisation indépendantes.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_NAME = "Attention-Enhanced-CNN-BiLSTM-Framework-for-Lithium-Ion-Battery-State-of-Health-Prediction"
repo = Path("/content") / REPO_NAME
if repo.is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/walidmchara/" + REPO_NAME + ".git", str(repo)], check=True)
os.chdir(repo)

for module, package in [
    ("scipy", "scipy"), ("sklearn", "scikit-learn"), ("tensorflow", "tensorflow"),
    ("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"), ("seaborn", "seaborn")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import tensorflow as tf
SEED = 42  # Initial notebook did not fix TensorFlow seeds; this makes reruns more controlled.
tf.keras.utils.set_random_seed(SEED)
EXPERIMENT = "B0018"
EPOCHS = 30
run_id = f"legacy_mat_{EXPERIMENT.lower()}_e{EPOCHS}_seed{SEED}"
run_dir = Path("/content/drive/MyDrive/NASA_CNN_BiLSTM_Attention") / run_id
checkpoint_dir = run_dir / "checkpoints"
output_dir = run_dir / "outputs"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# Original NASA archive is cached outside the GitHub repository.
archive = Path("/content/drive/MyDrive/NASA_original_source/5.+Battery+Data+Set.zip")
cmd = [sys.executable, "scripts/restore_original_nasa_mat.py",
       "--archive", str(archive), "--dest", "data",
       "--manifest-out", str(run_dir / "mat_source_provenance.json"),
       "--cells", "B0005", "B0006", "B0007", "B0018"]
subprocess.run(cmd, cwd=repo, check=True)
print("Original MATLAB inputs verified; legacy run:", run_id)
print("Google Drive recovery:", checkpoint_dir)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
import datetime
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def load_data(battery):
  mat = loadmat(str(next(p for p in [Path('data'), Path('../data')] if p.is_dir()) / (battery + '.mat')))
  print('Total data in dataset: ', len(mat[battery][0, 0]['cycle'][0]))
  counter = 0
  dataset = []
  capacity_data = []

  for i in range(len(mat[battery][0, 0]['cycle'][0])):
    row = mat[battery][0, 0]['cycle'][0, i]
    if row['type'][0] == 'discharge':
      ambient_temperature = row['ambient_temperature'][0][0]
      date_time = datetime.datetime(int(row['time'][0][0]),
                               int(row['time'][0][1]),
                               int(row['time'][0][2]),
                               int(row['time'][0][3]),
                               int(row['time'][0][4])) + datetime.timedelta(seconds=int(row['time'][0][5]))
      data = row['data']
      capacity = data[0][0]['Capacity'][0][0]
      for j in range(len(data[0][0]['Voltage_measured'][0])):
        voltage_measured = data[0][0]['Voltage_measured'][0][j]
        current_measured = data[0][0]['Current_measured'][0][j]
        temperature_measured = data[0][0]['Temperature_measured'][0][j]
        current_load = data[0][0]['Current_load'][0][j]
        voltage_load = data[0][0]['Voltage_load'][0][j]
        time = data[0][0]['Time'][0][j]
        dataset.append([counter + 1, ambient_temperature, date_time, capacity,
                        voltage_measured, current_measured,
                        temperature_measured, current_load,
                        voltage_load, time])
      capacity_data.append([counter + 1, ambient_temperature, date_time, capacity])
      counter = counter + 1
  print(dataset[0])
  return [pd.DataFrame(data=dataset,
                       columns=['cycle', 'ambient_temperature', 'datetime',
                                'capacity', 'voltage_measured',
                                'current_measured', 'temperature_measured',
                                'current_load', 'voltage_load', 'time']),
          pd.DataFrame(data=capacity_data,
                       columns=['cycle', 'ambient_temperature', 'datetime',
                                'capacity'])]

In [ ]:
dataset1, capacity1 = load_data('B0005')
dataset2, capacity2 = load_data('B0006')
dataset3, capacity3 = load_data('B0007')
dataset = pd.concat([dataset1, dataset2,dataset3], axis=0)
capacity = pd.concat([capacity1, capacity2,capacity3], axis=0)
capacity = capacity.reset_index(drop=True)
dataset = dataset.reset_index(drop=True)
pd.set_option('display.max_columns', 10)
print(dataset.head())
dataset.describe()

In [ ]:
attrib=['cycle', 'datetime', 'capacity']
dis_ele = capacity[attrib]
C = dis_ele['capacity'][0]
for i in range(len(dis_ele)):
    dis_ele['SoH']=(dis_ele['capacity'])/C
print(dis_ele)

C = dataset['capacity'][0]
soh = []
for i in range(len(dataset)):
  soh.append([dataset['capacity'][i] / C])
soh = pd.DataFrame(data=soh, columns=['SoH'])

attribs=['capacity', 'voltage_measured', 'current_measured',
         'temperature_measured', 'current_load', 'voltage_load', 'time']
train_dataset = dataset[attribs]
sc = MinMaxScaler(feature_range=(0,1))
train_dataset = sc.fit_transform(train_dataset)
print(train_dataset.shape)
print(soh.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(train_dataset, soh, test_size=0.2, random_state=0)
print('Train set shape', X_train.shape)
print('Validation set shape', X_valid.shape)

In [ ]:

X_train_series = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_valid_series = X_valid.reshape((X_valid.shape[0], X_valid.shape[1], 1))
print('Train set shape', X_train_series.shape)
print('Validation set shape', X_valid_series.shape)

In [ ]:
from tensorflow.keras.layers import Dense, Dropout, Flatten, LSTM, Bidirectional, Conv1D, MaxPooling1D, Multiply,Input
from tensorflow.keras.models import Model
window=2
inputs=Input(shape=(X_train_series.shape[1], X_train_series.shape[2]))
model=Conv1D(filters = 64, kernel_size = 3, activation = 'relu')(inputs)
model=MaxPooling1D(pool_size = window)(model)
model=Dropout(0.3)(model)
model=Bidirectional(LSTM(100, activation='relu'), name='bilstm')(model)
attention=Dense(200, activation='sigmoid', name='attention_vec')(model)
model=Multiply()([model, attention])
outputs = Dense(1, activation='relu')(model)
model = Model(inputs=inputs, outputs=outputs)
model.compile(loss='mse',optimizer='adam',metrics=['accuracy'])
model.summary()

In [ ]:
from scripts.legacy_mat_recovery import fit_exact_notebook_protocol, provenance_for_legacy
legacy_provenance = provenance_for_legacy(repo, EXPERIMENT, seed=SEED)
model, AM_CNN_lstm_history = fit_exact_notebook_protocol(
    model, X_train_series, Y_train, X_valid_series, Y_valid,
    checkpoint_dir=checkpoint_dir,
    signature_settings=legacy_provenance,
    epochs=EPOCHS,
)
print("Original architecture and sample-level protocol preserved.")

In [ ]:
print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> TESTING")
dataset_val, capacity_val = load_data('B0018')
attrib=['cycle', 'datetime', 'capacity']
dis_ele = capacity_val[attrib]
C = dis_ele['capacity'][0]
for i in range(len(dis_ele)):
    dis_ele['SoH']=(dis_ele['capacity']) / C
print(dataset_val.head(5))
print(dis_ele.head(5))
attrib=['capacity', 'voltage_measured', 'current_measured',
        'temperature_measured', 'current_load', 'voltage_load', 'time']
# Original sc.fit_transform(test) retained exactly; Keras 3 requires the
# additional singleton feature channel axis (N, 7, 1).
test_scaled = sc.fit_transform(dataset_val[attrib])
soh_pred = model.predict(test_scaled.reshape((test_scaled.shape[0], test_scaled.shape[1], 1)))
print(soh_pred.shape)
print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>.ACTUAL AND PREDICTED SOH")
C = dataset_val['capacity'][0]
soh = []
for i in range(len(dataset_val)):
  soh.append(dataset_val['capacity'][i] / C)
new_soh = dataset_val.loc[(dataset_val['cycle'] >= 1), ['cycle']]
new_soh['SoH'] =  soh
new_soh['NewSoH'] = soh_pred
new_soh = new_soh.groupby(['cycle']).mean().reset_index()
print(new_soh.head(10))
rms = np.sqrt(mean_squared_error(new_soh['SoH'], new_soh['NewSoH']))
print('Root Mean Square Error: ', rms)
# Calculate Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((new_soh['SoH'] - new_soh['NewSoH']) / new_soh['SoH'])) * 100
print("Mean Absolute Percentage Error (MAPE):", mape)
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(new_soh['SoH'], new_soh['NewSoH'])
print("Mean Squared Error (MSE):", mse)
mae = mean_absolute_error(new_soh['SoH'], new_soh['NewSoH'])
print(f'Mean Absolute Error (MAE): {mae:.4f}')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
# Tracer les valeurs réelles vs les valeurs prédites
plt.figure(figsize=(10, 6))
fig, ax = plt.subplots(1, figsize=(12, 8))
plt.plot(new_soh['SoH'], color='blue', linewidth=2, label='Actual-SoH', marker='o')
plt.plot(new_soh['NewSoH'], color='green', linewidth=2, label='Predicted-SoH', marker='o')
ax.set_xlabel('cycle', fontsize=22)  # Taille de police de l'axe des abscisses
ax.set_ylabel('SoH', fontsize=22)       # Taille de police de l'axe des ordonnées
ax.set_title('Actual vs Predicted SoH -AM-CNN-BILSTM B0018')
ax.tick_params(axis='x', which='major', labelsize=18)  # Taille des hachures des axes des abscisses
ax.tick_params(axis='y', which='major', labelsize=18)  # Taille des hachures des axes des ordonnées
# Légende
plt.legend(fontsize=24)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
plt.plot(AM_CNN_lstm_history.history['loss'], label='Train loss')
plt.plot(AM_CNN_lstm_history.history['val_loss'], label='Validation loss')
plt.legend(loc='best')
plt.title('AM CNN BILSTM  Learning Curve')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.show()

In [ ]:
# Same final-epoch weights; modern portable full-model format on Drive.
model.save(str(output_dir / "AM_CNN_BILSTM_B0018.keras"))
from tensorflow.keras.models import load_model
model = load_model(str(output_dir / "AM_CNN_BILSTM_B0018.keras"))
print("Final model saved:", output_dir / "AM_CNN_BILSTM_B0018.keras")

In [ ]:
# Save the exact historical cycle-aggregated test metrics and traceable protocol.
import json
from sklearn.metrics import r2_score
from scripts.legacy_mat_recovery import atomic_json

history_frame = pd.DataFrame(AM_CNN_lstm_history.history)
history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
history_frame.to_csv(output_dir / "training_history.csv", index=False)
new_soh.to_csv(output_dir / "test_predictions.csv", index=False)
score = {
    "experiment": EXPERIMENT,
    "protocol": "original-mat-legacy-with-target-leakage",
    "epochs_completed": len(history_frame),
    "evaluation": "last epoch; raw discharge samples averaged within cycle",
    "test": {
        "mae": float(mae), "rmse": float(rms), "mse": float(mse),
        "mape_percent": float(mape),
        "r2_additional": float(r2_score(new_soh["SoH"], new_soh["NewSoH"])),
        "discharge_cycles": int(len(new_soh)),
    },
    "scientific_warning": (
        "The original notebook inputs include same-cycle capacity (target leakage), "
        "normalizes test data with a new MinMaxScaler fit, and uses sample-level "
        "random validation. These values are not leakage-free generalization scores."
    ),
    "source": "Original user-provided MATLAB notebook experiment"
}
atomic_json(output_dir / "metrics.json", score)
atomic_json(output_dir / "config.json", legacy_provenance)
print(json.dumps(score, indent=2))
print("Saved original-notebook outputs in:", output_dir)

## Publication facultative dans GitHub

Le dossier `results/` peut contenir les métriques, courbes et prédictions de cette **reproduction historique explicitement identifiée comme présentant une fuite de cible**. Les `.mat` et les snapshots `.keras` restent sur Google Drive. Le secret Colab `GithubToken` doit avoir `Contents: Read and write` pour ce dépôt public. Ne jamais afficher sa valeur.

In [ ]:
from google.colab import userdata
token = userdata.get("GithubToken")
if not token:
    print("Token non configuré : résultats conservés dans Google Drive seulement.")
elif (repo / "results" / run_id / "manifest.json").exists():
    print("Cette reproduction a déjà été publiée :", run_id)
else:
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    subprocess.run([
        sys.executable, "scripts/publish_results.py",
        "--source", str(output_dir), "--run-id", run_id
    ], cwd=repo, env=env, check=True)
    print(f"https://github.com/walidmchara/{REPO_NAME}/tree/main/results/{run_id}")